# HyperQwen Eval: PWMV × LIMO LoRA Stacking

Tests if the 220-sample LIMO LoRA adapter stacks productively with PWMV wrapper.

**Target comparison**:

| method | previously measured | expected |
|---|---|---|
| base greedy | 0.700 (n=50) | — |
| base + PWMV | 0.760 (n=50) | — |
| adapter greedy | **new** | > 0.70 if LoRA helps |
| adapter + PWMV | **new** | > 0.76 if stack works |

**N_EVAL=15** (tight budget, directional signal). ~2h compute on RTX 6000.

**Same seed 123** as original PWMV eval to avoid cherry-picking.

## Cell 1 — Setup

In [ ]:
import sys, subprocess, os, shutil
from pathlib import Path
def pip(*a): return subprocess.run([sys.executable, '-m', 'pip', *a], check=False)

try:
    import transformers
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
    ok = 'qwen3_5_moe' in CONFIG_MAPPING_NAMES
except Exception: ok = False
if not ok:
    pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
        'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
        'protobuf', 'scipy', 'hf_transfer', 'peft')
    pip('uninstall', '-y', 'transformers', 'causal-conv1d')
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','--depth=1',
                    'https://github.com/huggingface/transformers.git', SRC], check=True)
    pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    pip('install','--no-cache-dir','flash-linear-attention')
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]

try:
    from google.colab import userdata
    t = userdata.get('HF_TOKEN')
    if t:
        from huggingface_hub import login
        login(token=t); print('HF auth OK')
except: pass

import json, re, time, random, pickle
import numpy as np
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from collections import defaultdict, Counter
from safetensors import safe_open
from huggingface_hub import snapshot_download, HfApi, create_repo
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

OUT = Path('/content/hyperqwen_eval'); OUT.mkdir(exist_ok=True)
HF_OUT = 'caiovicentino1/Qwen3.6-35B-A3B-LIMO-Probe'
HF_STAGE_B = 'caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b'
MODEL_ID = 'Qwen/Qwen3.6-35B-A3B'
ADAPTER_PATH = '/content/limo_lora/lora_adapter'

PROBE_LAYER = 11
N_TRACES = 4
TEMP = 0.7
MAX_NEW = 3500
N_EVAL = 15
FORCE_SUFFIX = '\n</think>\n\nFinal answer: \\boxed{'

print('setup done')

## Cell 2 — Load base model + LoRA adapter

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
from peft import PeftModel

tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map='cuda',
    attn_implementation='sdpa', trust_remote_code=True)
base_model.eval()
for p in base_model.parameters(): p.requires_grad_(False)
print(f'Base model: {torch.cuda.memory_allocated()/1e9:.1f} GB')

# Attach LoRA adapter
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print(f'After adapter: {torch.cuda.memory_allocated()/1e9:.1f} GB')

## Cell 3 — Hook L11 + load probe

In [ ]:
captured_l11 = {'chunks': []}

def l11_hook(module, inp, out):
    h = out[0] if isinstance(out, tuple) else out
    captured_l11['chunks'].append(h[:, -1, :].detach().float().cpu())

# PEFT wraps base_model inside model.base_model
# Access L11 through the base
base_ref = model.base_model if hasattr(model, 'base_model') else model
if hasattr(base_ref, 'model') and hasattr(base_ref.model, 'language_model'):
    layer_L = base_ref.model.language_model.layers[PROBE_LAYER]
elif hasattr(base_ref, 'model') and hasattr(base_ref.model, 'model'):
    layer_L = base_ref.model.model.layers[PROBE_LAYER]
else:
    # Fallback traversal
    def find_layer(m, idx):
        for nm, cc in m.named_children():
            if 'layer' in nm and hasattr(cc, '__getitem__'):
                try: return cc[idx]
                except: pass
            r = find_layer(cc, idx)
            if r is not None: return r
        return None
    layer_L = find_layer(base_ref, PROBE_LAYER)

assert layer_L is not None, 'Could not find L11'
h_handle = layer_L.register_forward_hook(l11_hook)
print(f'OK L{PROBE_LAYER} hook installed on {type(layer_L).__name__}')

# Load probe from deepconf-probe repo
probe_repo = snapshot_download('caiovicentino1/qwen36-deepconf-probe',
                                repo_type='model', cache_dir='/content/cache',
                                allow_patterns=['probe_l11.pkl'])
with open(Path(probe_repo) / 'probe_l11.pkl', 'rb') as f:
    probe = pickle.load(f)
print(f'OK probe loaded')

## Cell 4 — Helpers + load eval prompts (same seed=123 as original)

In [ ]:
def extract_answer(text):
    post = text.split('</think>')[-1] if '</think>' in text else text
    for pattern in [
        r'\\boxed\{([A-J])\}',
        r'\\boxed\s*\{?\s*([A-J])\s*\}?',
        r'[Aa]nswer\s*(?:is\s*)?[:=]?\s*\*?\*?\(?([A-J])\)?\*?\*?',
        r'[Tt]he\s+(?:correct\s+)?answer\s+is\s+\(?([A-J])\)?',
        r'^\s*\(?([A-J])\)?\s*\.?\s*$',
    ]:
        m = re.search(pattern, post, re.MULTILINE)
        if m: return m.group(1).upper()
    m = re.findall(r'\\boxed\{([A-J])\}', text)
    if m: return m[-1]
    tail = post[-300:] if post else text[-300:]
    letters = re.findall(r'\b([A-J])\b', tail)
    return letters[-1] if letters else None

def format_prompt(q, opts):
    choices = '\n'.join(f'{chr(65+i)}. {o}' for i, o in enumerate(opts))
    content = ('Answer the following multiple-choice question. '
        'Reason step by step, then put the letter in \\boxed{}.\n\n'
        f'Question: {q}\n\nOptions:\n{choices}')
    return tok.apply_chat_template([{'role':'user','content':content}],
                                    tokenize=False, add_generation_prompt=True,
                                    enable_thinking=True)

def _force_close(full_ids, prompt_len):
    gen_text = tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=False)
    if '</think>' in gen_text:
        return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True)
    full_text = tok.decode(full_ids.tolist(), skip_special_tokens=False) + FORCE_SUFFIX
    ctx_ids = tok(full_text, return_tensors='pt', add_special_tokens=False).input_ids.cuda()
    with torch.no_grad():
        out = model.generate(ctx_ids, max_new_tokens=15, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    suffix = tok.decode(out[0, ctx_ids.shape[1]:].tolist(), skip_special_tokens=True)
    return tok.decode(full_ids[prompt_len:].tolist(), skip_special_tokens=True) + FORCE_SUFFIX + suffix

# Load eval prompts — SAME seed 123 as original PWMV eval
corpus = snapshot_download(HF_STAGE_B, repo_type='dataset', cache_dir='/content/cache')
shards = sorted((Path(corpus)/'shards').glob('q*.safetensors'))

questions = []
seen = set()
for shard in shards:
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        q = meta['question']
        if q in seen: continue
        seen.add(q)
        opts = json.loads(meta['options'])
        if len(opts) != 10: continue
        questions.append(dict(question=q, options=opts, gold_letter=meta['gold_letter']))

random.seed(123)
random.shuffle(questions)
eval_set = questions[:N_EVAL]
print(f'{len(eval_set)} eval prompts (same seed 123 as original)')

## Cell 5 — Generation wrappers (greedy + PWMV)

In [ ]:
def generate_greedy(ids):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NEW, do_sample=False,
                             pad_token_id=tok.pad_token_id, use_cache=True)
    text = _force_close(out[0], ids.shape[1])
    return extract_answer(text)

def generate_pwmv(ids, n_traces=N_TRACES, temp=TEMP):
    captured_l11['chunks'] = []
    with torch.no_grad():
        out = model.generate(
            ids, max_new_tokens=MAX_NEW, do_sample=True, temperature=temp,
            num_return_sequences=n_traces, top_p=0.95,
            pad_token_id=tok.pad_token_id, use_cache=True)
    chunks = captured_l11['chunks']
    if len(chunks) >= 2:
        gen_stack = torch.stack(chunks[1:], dim=0)
        trace_embs = gen_stack.mean(dim=0).numpy()
        probe_scores = probe.predict_proba(trace_embs)[:, 1].tolist()
    else:
        probe_scores = [1.0] * n_traces
    answers = []
    for i in range(n_traces):
        text = _force_close(out[i], ids.shape[1])
        answers.append(extract_answer(text))
    scores = defaultdict(float)
    for a, w in zip(answers, probe_scores):
        if a: scores[a] += w
    return (max(scores, key=scores.get) if scores else None), answers, probe_scores

print('Helpers ready')

## Cell 6 — Run 4 conditions: base/adapter × greedy/PWMV

Using `model.disable_adapter()` context to toggle LoRA on/off without reload. 4 configs per prompt = 6 forwards total (greedy×2 + PWMV×2).

In [ ]:
results = {'base_greedy': [], 'adapter_greedy': [], 'base_pwmv': [], 'adapter_pwmv': []}
details = {'base_pwmv': [], 'adapter_pwmv': []}

t0 = time.time()
for qi, q in enumerate(tqdm(eval_set, desc='eval')):
    p = format_prompt(q['question'], q['options'])
    ids = tok(p, return_tensors='pt').input_ids.cuda()
    if ids.shape[1] > 1536: continue
    gold = q['gold_letter']

    # ADAPTER paths (default)
    try:
        ag = generate_greedy(ids)
        results['adapter_greedy'].append((ag, ag == gold if ag else False))
    except Exception as e:
        print(f'  adapter_greedy err: {e}')
        results['adapter_greedy'].append((None, False))
    try:
        ap, a_answers, a_scores = generate_pwmv(ids)
        results['adapter_pwmv'].append((ap, ap == gold if ap else False))
        details['adapter_pwmv'].append({'gold': gold, 'pred': ap, 'traces': a_answers, 'scores': a_scores})
    except Exception as e:
        print(f'  adapter_pwmv err: {e}')
        results['adapter_pwmv'].append((None, False))

    # BASE paths (adapter disabled)
    with model.disable_adapter():
        try:
            bg = generate_greedy(ids)
            results['base_greedy'].append((bg, bg == gold if bg else False))
        except Exception as e:
            results['base_greedy'].append((None, False))
        try:
            bp, b_answers, b_scores = generate_pwmv(ids)
            results['base_pwmv'].append((bp, bp == gold if bp else False))
            details['base_pwmv'].append({'gold': gold, 'pred': bp, 'traces': b_answers, 'scores': b_scores})
        except Exception as e:
            results['base_pwmv'].append((None, False))

    # Periodic
    if (qi+1) % 3 == 0:
        print(f'  [{qi+1}/{len(eval_set)}]', end='  ')
        for name in ['base_greedy', 'adapter_greedy', 'base_pwmv', 'adapter_pwmv']:
            acc = np.mean([r[1] for r in results[name]])
            print(f'{name}={acc:.3f}', end='  ')
        print()

h_handle.remove()

print(f'\n=== FINAL (n={len(eval_set)} prompts, {(time.time()-t0)/60:.1f} min) ===')
for name in ['base_greedy', 'base_pwmv', 'adapter_greedy', 'adapter_pwmv']:
    acc = np.mean([r[1] for r in results[name]])
    none_rate = sum(1 for r in results[name] if r[0] is None) / max(1, len(results[name]))
    print(f'{name:20s}: accuracy = {acc:.3f}  (None rate: {none_rate*100:.0f}%)')

# Stacking delta
b_g = np.mean([r[1] for r in results['base_greedy']])
b_p = np.mean([r[1] for r in results['base_pwmv']])
a_g = np.mean([r[1] for r in results['adapter_greedy']])
a_p = np.mean([r[1] for r in results['adapter_pwmv']])

print(f'\n=== DELTAS ===')
print(f'PWMV lift (base):      {b_p - b_g:+.3f}')
print(f'PWMV lift (adapter):   {a_p - a_g:+.3f}')
print(f'LoRA lift (greedy):    {a_g - b_g:+.3f}')
print(f'LoRA lift (PWMV):      {a_p - b_p:+.3f}')
print(f'STACK (both combined): {a_p - b_g:+.3f} vs base greedy')

## Cell 7 — Save + upload eval results

In [ ]:
summary = {
    'model': MODEL_ID,
    'adapter_path': ADAPTER_PATH,
    'n_eval_prompts': len(eval_set),
    'eval_seed': 123,
    'n_traces': N_TRACES,
    'temperature': TEMP,
    'max_new_tokens': MAX_NEW,
    'results': {
        name: float(np.mean([r[1] for r in results[name]]))
        for name in results
    },
    'deltas': {
        'pwmv_on_base_pp': float((b_p - b_g) * 100),
        'pwmv_on_adapter_pp': float((a_p - a_g) * 100),
        'lora_on_greedy_pp': float((a_g - b_g) * 100),
        'lora_on_pwmv_pp': float((a_p - b_p) * 100),
        'full_stack_pp': float((a_p - b_g) * 100),
    },
}
with open(OUT / 'hyperqwen_eval_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
with open(OUT / 'pwmv_details.json', 'w') as f:
    json.dump(details, f, indent=2)

api = HfApi()
for fn in ['hyperqwen_eval_summary.json', 'pwmv_details.json']:
    api.upload_file(path_or_fileobj=str(OUT/fn), path_in_repo=fn,
                    repo_id=HF_OUT, repo_type='model',
                    commit_message=f'HyperQwen eval n={len(eval_set)}: {fn}')
    print(f'OK {fn}')
print(f'\nDone: https://huggingface.co/{HF_OUT}')